# QKD Information Reconciliation — Benchmark Sweep on Kaggle

This notebook runs the Phase 1 benchmark for the `qkd-cascade-ldpc` project:
Cascade vs. Mueller blind LDPC vs. Borisov adaptive LDPC across a QBER
sweep and a QBER-mismatch sweep, then renders the 7 slide-deck figures.

**Use Kaggle CPU (no GPU).** The bottlenecks are Python loops + SeQUeNCe
discrete-event simulation — GPU gives no speedup. Parallelism comes from
`multiprocessing.Pool` over the 4 CPU cores.

Default config: n=8192, 200 frames/Q ≈ **30 min wall-clock**. Adjust in Cell 4.

## Cell 1 — Install dependencies

In [ ]:
!pip install sequence pyarrow -q

## Cell 2 — Clone the repo (Option A)

In [ ]:
import os, sys, subprocess

REPO_URL = "https://github.com/alexandrachirita98/qkd-cascade-ldpc.git"
REPO_DIR = "/kaggle/working/qkd-cascade-ldpc"

if not os.path.exists(REPO_DIR):
    print(f"cloning {REPO_URL} → {REPO_DIR}...")
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
else:
    print(f"repo already at {REPO_DIR}; pulling latest...")
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=False)

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
print(f"cwd        : {os.getcwd()}")
print(f"sys.path[0]: {sys.path[0]}")

## Cell 3 — Verify the LDPC code pool

In [ ]:
from src.codes.storage import list_available

pool = list_available()
print(f"Available codes: {len(pool)}")
for n, r in pool:
    print(f"  n={n:>5}  R={r:.2f}")

n8192 = sum(1 for n, _ in pool if n == 8192)
if n8192 < 9:
    print(f"\n[!] only {n8192}/9 n=8192 codes; generating (~5 min)...")
    !python -m src.tools.generate_codes --frame-lengths 8192
    pool = list_available()
    print(f"\nAfter: {sum(1 for n,_ in pool if n==8192)} n=8192 codes")

## Cell 4 — Sweep parameters

Adjust `N`, `FRAMES_PER_Q`, and `N_WORKERS` here. Wall-clock estimate is
printed below — sanity-check it before running Cell 5.

In [ ]:
N             = 8192      # 1024 (fast) or 8192 (canonical)
ALPHA         = 0.15
FRAMES_PER_Q  = 200       # 1000 = canonical; 200 = ~30 min on Kaggle CPU
N_WORKERS     = 4         # Kaggle CPU = 4 cores
SEED          = 42

payload = N - round(ALPHA * N)
QBERS = [round(0.005 * (i + 1), 4) for i in range(20)]   # 0.005..0.100, step 0.005

n_frames_total = len(QBERS) * FRAMES_PER_Q * 3
est_min = n_frames_total * 0.08 / N_WORKERS / 60

print(f"N            = {N}")
print(f"alpha        = {ALPHA}")
print(f"payload/frame= {payload}")
print(f"frames/Q     = {FRAMES_PER_Q}")
print(f"workers      = {N_WORKERS}")
print(f"QBER points  = {len(QBERS)} from {QBERS[0]} to {QBERS[-1]}")
print(f"Total frames : {n_frames_total}")
print(f"Estimated wall-clock: ~{est_min:.0f} min")

## Cell 5 — Parallel QBER sweep

One worker per QBER point. Each worker builds its own algorithm instances
and generates its own SeQUeNCe frames, so there's no shared state to lock.
Uses `fork` (the Linux default on Kaggle) so worker functions and imports
are inherited from the parent.

If you re-run this cell, the previous parquet is overwritten.

In [ ]:
import multiprocessing as mp
import pandas as pd
import time

def _one_qber_point(args):
    """Process one Q point with all three algorithms; return a partial DataFrame."""
    import os, sys
    sys.path.insert(0, "/kaggle/working/qkd-cascade-ldpc")
    os.chdir("/kaggle/working/qkd-cascade-ldpc")
    from src.harness import make_cascade, make_mueller, make_borisov, generate_frames

    q, n_payload, n_frames, seed, n, alpha = args
    algos = [
        make_cascade(seed=seed),
        make_mueller(n=n, seed=seed),
        make_borisov(n=n, alpha=alpha, seed=seed),
    ]
    frames = generate_frames(q, n_payload, n_frames, seed=seed)
    rows = []
    for alg in algos:
        for i, (a, b, true_q) in enumerate(frames):
            res = alg.fn(a, b, q, true_q)
            rows.append({
                "q": q, "alg": alg.name, "frame_idx": i,
                "leakage_bits": res.leakage_bits, "messages": res.messages,
                "iterations": res.iterations, "success": res.success,
                "wall_clock_s": res.wall_clock_s, "true_qber": res.true_qber,
            })
    return pd.DataFrame(rows)

work = [(q, payload, FRAMES_PER_Q, SEED, N, ALPHA) for q in QBERS]
t0 = time.perf_counter()
print(f"QBER sweep: {len(work)} points on {N_WORKERS} workers...\n")

with mp.get_context("fork").Pool(processes=N_WORKERS) as pool:
    parts = []
    for i, df_part in enumerate(pool.imap_unordered(_one_qber_point, work)):
        parts.append(df_part)
        q_done = df_part["q"].iloc[0]
        print(f"  [{i+1:>2}/{len(work)}] Q={q_done:.3f} done  "
              f"({(time.perf_counter()-t0)/60:.1f} min elapsed)")

df_q = pd.concat(parts, ignore_index=True)
df_q.to_parquet("/kaggle/working/qber_sweep.parquet", index=False)
print(f"\nQBER sweep total: {(time.perf_counter()-t0)/60:.1f} min; "
      f"{len(df_q)} records → /kaggle/working/qber_sweep.parquet")

## Cell 6 — Mismatch sweep (serial)

Smaller workload (3 trueQ × 5 Δ × FRAMES_PER_Q × 3 alg). Running serial
keeps it simple; parallelization would only save a few minutes here.

In [ ]:
from src.harness import run_mismatch_sweep, make_cascade, make_mueller, make_borisov

algorithms = [
    make_cascade(seed=SEED),
    make_mueller(n=N, seed=SEED),
    make_borisov(n=N, alpha=ALPHA, seed=SEED),
]

print("mismatch sweep (serial)...\n")
t0 = time.perf_counter()
df_m = run_mismatch_sweep(
    algorithms,
    true_qbers=[0.02, 0.04, 0.06],
    deltas=[-0.02, -0.01, 0.0, 0.01, 0.02],
    n_frames_per_point=FRAMES_PER_Q,
    n_payload=payload,
    seed=SEED,
    progress_callback=lambda m: print(f"  {m}"),
)
df_m.to_parquet("/kaggle/working/mismatch_sweep.parquet", index=False)
print(f"\nmismatch sweep total: {(time.perf_counter()-t0)/60:.1f} min; "
      f"{len(df_m)} records → /kaggle/working/mismatch_sweep.parquet")

## Cell 7 — Render and display the 7 slide-deck plots

In [ ]:
from src.harness import make_slide_deck
from pathlib import Path
from IPython.display import Image, display, Markdown

out_dir = Path("/kaggle/working/plots")
paths = make_slide_deck(df_q, df_m, n_payload=payload, out_dir=out_dir)

print(f"rendered {len(paths)} plots → {out_dir}/\n")
for name, p in paths.items():
    display(Markdown(f"### `{name}`"))
    display(Image(str(p)))

## Cell 8 — Quick summary table

Per-(Q, algorithm) headline metrics. Handy as a sanity check before
downloading the figures.

In [ ]:
from src.harness import per_qa_summary

summary = per_qa_summary(df_q, n_payload=payload)
cols = ["alg", "q", "FER", "f", "f_eff", "mean_messages", "mean_wall_ms", "R_sec_per_block"]
display(summary[cols].round(4))

## Cell 9 — Bundle artifacts for download

In [ ]:
import shutil
zip_path = shutil.make_archive(
    "/kaggle/working/qkd_sweep_results", "zip", "/kaggle/working",
)
size_mb = os.path.getsize(zip_path) / 1024 / 1024
print(f"created {zip_path} ({size_mb:.1f} MB)\n")
print("Download from the Kaggle Notebook → 'Output' tab on the right side")
print("of the notebook view (look for 'qkd_sweep_results.zip').")